# SAUVC 2027 — RF-DETR Large 訓練（資料集 v2）

**流程**：掛載 Drive → 安裝 → 把資料從 Drive 複製到 Colab 本機 → 檢查資料 →（選擇性）煙霧測試 → 正式訓練 → 評估 → 視覺化 → 匯出 ONNX

**Drive 上的位置**（都在「我的雲端硬碟」）：

| 路徑 | 用途 |
|---|---|
| `SAUVC_RFDETR/data/sauvc_rfdetr_v2.zip` | 訓練資料（Roboflow COCO 格式） |
| `SAUVC_RFDETR/checkpoints/<RUN_NAME>/` | 訓練輸出：checkpoint、log、評估結果、ONNX |
| `SAUVC_RFDETR/notebooks/` | 這份 notebook |

**開始前**：Colab 選單「執行階段 → 變更執行階段類型」選 GPU（A100 / L4 / T4 都可以）。

**斷線怎麼辦**：每個 epoch 都會把 `last.ckpt` 寫進 Drive（覆寫）。重新連線後，從第 1 格依序執行到「正式訓練」，`RUN_NAME` 不要改，就會自動從最後的 checkpoint 接著跑。

## 1. 設定

In [ ]:
# ===== 只需要改這一格 =====
RUN_NAME   = "v2_large_e30"   # 每次新的實驗換一個名字；同名會接著上次的 checkpoint 續跑
EPOCHS     = 30               # 資料約 2.5 萬張，先 30 個 epoch，看 valid 曲線再決定要不要加
LR         = 1e-4
SMOKE_TEST = False            # True = 只用 200 張跑 1 個 epoch，確認流程正常（約幾分鐘）

DRIVE_ROOT = "/content/drive/MyDrive/SAUVC_RFDETR"
ZIP_PATH   = f"{DRIVE_ROOT}/data/sauvc_rfdetr_v2.zip"
CKPT_ROOT  = f"{DRIVE_ROOT}/checkpoints"
LOCAL_DATA = "/content/sauvc_rfdetr_v2"   # 解壓到 Colab 本機，讀取比 Drive 快很多

## 2. 掛載 Drive、確認 GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
assert os.path.exists(ZIP_PATH), f"找不到 {ZIP_PATH}，請先把 sauvc_rfdetr_v2.zip 放到 Drive 的 SAUVC_RFDETR/data/"
print("zip:", round(os.path.getsize(ZIP_PATH) / 1e9, 2), "GB")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 3. 安裝

In [ ]:
!pip install -q "rfdetr[train,onnx]==1.12.0" tensorboard pycocotools supervision
import rfdetr, torch
print("rfdetr", getattr(rfdetr, "__version__", "?"), "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

## 4. 把資料複製到 Colab 本機並解壓

已經解壓過（同一個執行階段）就會跳過。

In [ ]:
import shutil, zipfile, time
if not os.path.exists(f"{LOCAL_DATA}/train/_annotations.coco.json"):
    t = time.time()
    shutil.copy(ZIP_PATH, "/content/data.zip")
    print(f"複製完成 {time.time() - t:.0f}s")
    with zipfile.ZipFile("/content/data.zip") as z:
        z.extractall("/content")
    os.remove("/content/data.zip")
    print(f"解壓完成 {time.time() - t:.0f}s")
print(sorted(os.listdir(LOCAL_DATA)))

## 5. 檢查資料

確認三個 split 都在、類別名稱正確、影像檔都存在。

In [ ]:
import json
from collections import Counter

def load_split(root, split):
    return json.load(open(f"{root}/{split}/_annotations.coco.json"))

for split in ["train", "valid", "test"]:
    c = load_split(LOCAL_DATA, split)
    names = {x["id"]: x["name"] for x in c["categories"]}
    missing = sum(not os.path.exists(f"{LOCAL_DATA}/{split}/{i['file_name']}") for i in c["images"])
    per_cls = Counter(names[a["category_id"]] for a in c["annotations"])
    print(f"[{split}] images {len(c['images'])}, boxes {len(c['annotations'])}, missing files {missing}")
    print("   ", dict(per_cls.most_common()))
print("categories:", [x["name"] for x in c["categories"]])

## 6.（選擇性）煙霧測試

`SMOKE_TEST = True` 時，從每個 split 抽一小部分做成迷你資料集，跑 1 個 epoch，確認安裝、資料格式和存檔都正常。正式訓練前建議先跑一次。

In [ ]:
import random

def make_subset(src, dst, n):
    for split, k in n.items():
        c = load_split(src, split)
        random.seed(0)
        imgs = random.sample(c["images"], min(k, len(c["images"])))
        keep = {i["id"] for i in imgs}
        os.makedirs(f"{dst}/{split}", exist_ok=True)
        for i in imgs:
            if not os.path.exists(f"{dst}/{split}/{i['file_name']}"):
                os.symlink(os.path.abspath(f"{src}/{split}/{i['file_name']}"), f"{dst}/{split}/{i['file_name']}")
        sub = dict(c, images=imgs, annotations=[a for a in c["annotations"] if a["image_id"] in keep])
        json.dump(sub, open(f"{dst}/{split}/_annotations.coco.json", "w"))

if SMOKE_TEST:
    DATA_DIR = "/content/smoke_data"
    make_subset(LOCAL_DATA, DATA_DIR, {"train": 200, "valid": 50, "test": 50})
    RUN, EPOCHS_TO_RUN = RUN_NAME + "_smoke", 1
else:
    DATA_DIR, RUN, EPOCHS_TO_RUN = LOCAL_DATA, RUN_NAME, EPOCHS
OUTPUT_DIR = f"{CKPT_ROOT}/{RUN}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("data:", DATA_DIR, "| output:", OUTPUT_DIR, "| epochs:", EPOCHS_TO_RUN)

## 7. 訓練

- batch size 依 GPU 顯存自動決定，總有效 batch 固定是 16（batch_size × grad_accum_steps）。如果 CUDA out of memory，把 `BATCH` 減半、`ACCUM` 加倍再跑。
- `OUTPUT_DIR` 裡已有 `last.ckpt`（每個 epoch 覆寫一次）時會自動續跑。
- 最好的模型存成 `checkpoint_best_total.pth`；完整的 epoch 存檔只在最後留一份，避免塞滿 Drive。

In [ ]:
from rfdetr import RFDETRLarge

mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if mem_gb >= 70:   BATCH, ACCUM = 16, 1   # A100 80GB / H100
elif mem_gb >= 35: BATCH, ACCUM = 8, 2    # A100 40GB
elif mem_gb >= 20: BATCH, ACCUM = 4, 4    # L4 24GB
else:              BATCH, ACCUM = 2, 8    # T4 16GB
print(f"GPU {torch.cuda.get_device_name(0)} {mem_gb:.0f} GB -> batch {BATCH} x accum {ACCUM}")

resume = next((p for p in [f"{OUTPUT_DIR}/last.ckpt", f"{OUTPUT_DIR}/checkpoint.pth"] if os.path.exists(p)), None)
print("resume from:", resume)

json.dump({"run": RUN, "epochs": EPOCHS_TO_RUN, "lr": LR, "batch": BATCH, "accum": ACCUM,
           "data": "sauvc_rfdetr_v2", "gpu": torch.cuda.get_device_name(0)},
          open(f"{OUTPUT_DIR}/run_config.json", "w"), indent=1)

model = RFDETRLarge()
train_kwargs = dict(dataset_dir=DATA_DIR, epochs=EPOCHS_TO_RUN, batch_size=BATCH,
                    grad_accum_steps=ACCUM, lr=LR, output_dir=OUTPUT_DIR, tensorboard=True,
                    checkpoint_interval=EPOCHS_TO_RUN)   # last.ckpt 每個 epoch 都會存；完整存檔只在最後一個 epoch 留一份
if resume:
    train_kwargs["resume"] = resume
model.train(**train_kwargs)
print("輸出檔案:", sorted(os.listdir(OUTPUT_DIR)))

### 訓練曲線（TensorBoard）

訓練中也可以另開一格執行，看 loss 和 valid mAP。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$OUTPUT_DIR"

## 8. 評估：每個類別的 AP，以及 gate / qualification_gate 互相誤判

用最好的 checkpoint，在 test 和 valid 上算 COCO AP（pycocotools）。結果也存到 `OUTPUT_DIR/eval_<split>.json`。

In [ ]:
import glob, numpy as np
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import contextlib, io

cands = [f"{OUTPUT_DIR}/{n}" for n in ["checkpoint_best_total.pth", "checkpoint_best_ema.pth", "checkpoint_best_regular.pth", "last.ckpt"]]
BEST = next(p for p in cands if os.path.exists(p))
print("best checkpoint:", BEST)
best_model = RFDETRLarge(pretrain_weights=BEST)

def model_names(m, coco_cats):
    names = getattr(m, "class_names", None)
    if isinstance(names, dict):
        return {int(k): v for k, v in names.items()}
    if isinstance(names, (list, tuple)):
        return dict(enumerate(names))
    return {c["id"]: c["name"] for c in coco_cats}   # fallback: same id space as the COCO file

def evaluate(split, score_thr=0.001, max_images=None):
    gt = COCO(f"{DATA_DIR}/{split}/_annotations.coco.json")
    cats = gt.loadCats(gt.getCatIds())
    name2id = {c["name"]: c["id"] for c in cats}
    id2name = model_names(best_model, cats)
    print("model class names:", id2name)
    results = []
    img_ids = gt.getImgIds()[:max_images] if max_images else gt.getImgIds()
    for k, iid in enumerate(img_ids):
        info = gt.loadImgs(iid)[0]
        det = best_model.predict(Image.open(f"{DATA_DIR}/{split}/{info['file_name']}").convert("RGB"), threshold=score_thr)
        for (x1, y1, x2, y2), cid, s in zip(det.xyxy, det.class_id, det.confidence):
            name = id2name.get(int(cid))
            if name in name2id:
                results.append({"image_id": iid, "category_id": name2id[name],
                                 "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(s)})
        if k % 300 == 0:
            print(f"  {split}: {k}/{len(img_ids)}")
    dt = gt.loadRes(results) if results else None
    out = {}
    for c in cats:
        if c["name"] == "sauvc-objects" or not gt.getAnnIds(catIds=[c["id"]]):
            continue
        E = COCOeval(gt, dt, "bbox"); E.params.catIds = [c["id"]]; E.params.imgIds = img_ids
        with contextlib.redirect_stdout(io.StringIO()):
            E.evaluate(); E.accumulate(); E.summarize()
        out[c["name"]] = {"AP50:95": round(E.stats[0], 3), "AP50": round(E.stats[1], 3),
                          "boxes": len(gt.getAnnIds(catIds=[c["id"]], imgIds=img_ids))}
    E = COCOeval(gt, dt, "bbox"); E.params.imgIds = img_ids
    E.params.catIds = [c["id"] for c in cats if c["name"] != "sauvc-objects"]
    with contextlib.redirect_stdout(io.StringIO()):
        E.evaluate(); E.accumulate(); E.summarize()
    out["ALL"] = {"AP50:95": round(E.stats[0], 3), "AP50": round(E.stats[1], 3)}
    return out, gt, results

def gate_confusion(gt, results, score=0.3, iou_thr=0.5):
    # for every gate / qualification_gate box: which class did the best-overlapping prediction get?
    names = {c["id"]: c["name"] for c in gt.loadCats(gt.getCatIds())}
    preds = {}
    for r in results:
        if r["score"] >= score:
            preds.setdefault(r["image_id"], []).append(r)
    def iou(a, b):
        ax2, ay2, bx2, by2 = a[0] + a[2], a[1] + a[3], b[0] + b[2], b[1] + b[3]
        iw, ih = max(0, min(ax2, bx2) - max(a[0], b[0])), max(0, min(ay2, by2) - max(a[1], b[1]))
        inter = iw * ih
        return inter / (a[2] * a[3] + b[2] * b[3] - inter + 1e-9)
    table = Counter()
    for a in gt.loadAnns(gt.getAnnIds()):
        g = names[a["category_id"]]
        if g not in ("gate", "qualification_gate"):
            continue
        best = max(preds.get(a["image_id"], []), key=lambda r: iou(a["bbox"], r["bbox"]), default=None)
        hit = names[best["category_id"]] if best and iou(a["bbox"], best["bbox"]) >= iou_thr else "missed"
        table[(g, hit)] += 1
    return table

for split in ["test", "valid"]:
    res, gt, dets = evaluate(split)
    json.dump(res, open(f"{OUTPUT_DIR}/eval_{split}.json", "w"), indent=1)
    print(f"\n===== {split} =====")
    for k, v in res.items():
        print(f"{k:20s} {v}")
    print("gate / qualification_gate (true -> predicted):", dict(gate_confusion(gt, dets)))

## 9. 看預測結果

從 test 隨機抽 8 張，畫出預測框（信心 ≥ 0.4）。

In [ ]:
import supervision as sv
import matplotlib.pyplot as plt

gt = COCO(f"{DATA_DIR}/test/_annotations.coco.json")
id2name = model_names(best_model, gt.loadCats(gt.getCatIds()))
random.seed(1)
infos = random.sample(gt.loadImgs(gt.getImgIds()), 8)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, info in zip(axes.flat, infos):
    img = Image.open(f"{DATA_DIR}/test/{info['file_name']}").convert("RGB")
    det = best_model.predict(img, threshold=0.4)
    labels = [f"{id2name.get(int(c), c)} {s:.2f}" for c, s in zip(det.class_id, det.confidence)]
    frame = sv.BoxAnnotator().annotate(np.array(img).copy(), det)
    frame = sv.LabelAnnotator().annotate(frame, det, labels)
    ax.imshow(frame); ax.set_title(info["file_name"][:40], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 10. 匯出 ONNX（給 Jetson Orin NX 轉 TensorRT）

ONNX 存到 `OUTPUT_DIR/export/`。到 Orin NX 上再用 `trtexec --onnx=... --fp16` 轉成 TensorRT engine。

In [ ]:
EXPORT_DIR = f"{OUTPUT_DIR}/export"
os.makedirs(EXPORT_DIR, exist_ok=True)
try:
    best_model.export(output_dir=EXPORT_DIR)
except TypeError:
    best_model.export()                                  # 舊版 rfdetr 沒有 output_dir 參數，會輸出到目前目錄
    for f in glob.glob("output/*.onnx") + glob.glob("*.onnx"):
        shutil.copy(f, EXPORT_DIR)
print(sorted(os.listdir(EXPORT_DIR)))